# HUẤN LUYỆN PHOBERT-BASE PHÂN LOẠI CẢM XÚC UIT CONFESSIONS (2022 - 2026)
**Nghiên cứu theo tiêu chuẩn IMRaD & Hướng 4 (Deep Learning / Transformers)**
- **Backbone**: `vinai/phobert-base`
- **Giải pháp cốt lõi**: Khắc phục mất cân bằng lớp bằng *Cost-Sensitive Class-Weighted Cross-Entropy Loss*
- **Mục tiêu**: Nâng Macro-F1 từ **75.23% (LinearSVC)** lên **>= 85.0% (SOTA Transformers)**
- **Phân chia dữ liệu**: Chuẩn Stratified 70% Train - 15% Validation - 15% Test

In [ ]:
# [BƯỚC 1] Cài đặt các thư viện cần thiết trên Kaggle (GPU Accelerator: T4 hoặc P100)
!pip install -q transformers accelerate pyvi evaluate scikit-learn seaborn
import torch
print(f"CUDA Có sẵn: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Đang sử dụng: {torch.cuda.get_device_name(0)}")
else:
    print("CẢNH BÁO: Hãy bật Accelerator -> GPU T4 x2 trong mục Settings bên phải!")

In [ ]:
# [BƯỚC 2] Tự động quét và nạp dữ liệu từ mọi thư mục trong /kaggle/input
import os
import glob
import pandas as pd
import numpy as np

# 1. Quét đệ quy toàn bộ thư mục /kaggle/input để tìm tất cả các file .csv
csv_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

# Quét thêm thư mục hiện tại nếu có
for f in glob.glob('*.csv'):
    if f not in csv_files:
        csv_files.append(f)

print("Danh sách file CSV tìm thấy trên hệ thống:")
for f in csv_files:
    print(" -", f)

# 2. Chọn file confession phù hợp nhất
csv_path = None
for candidate in csv_files:
    base = os.path.basename(candidate).lower()
    if any(k in base for k in ['clean', 'confession', 'sentiment', 'uit']):
        csv_path = candidate
        break

# Nếu không khớp từ khóa thì lấy file CSV đầu tiên
if csv_path is None and len(csv_files) > 0:
    csv_path = csv_files[0]

if csv_path is None:
    raise FileNotFoundError("Không tìm thấy file .csv nào trong /kaggle/input! Vui lòng kiểm tra lại dataset đã thêm.")

print(f"\n-> Đang nạp dữ liệu từ tệp: {csv_path}")
df = pd.read_csv(csv_path)
print(f"-> Tổng số confession: {len(df)}")
print(f"-> Các cột trong file: {list(df.columns)}")

# Nhận diện hoặc tự động gán nhãn cảm xúc
label_col = 'sentiment_lexicon' if 'sentiment_lexicon' in df.columns else ('sentiment' if 'sentiment' in df.columns else None)

if label_col is None:
    print("\n-> File chưa có cột nhãn 'sentiment_lexicon', đang tự động gán nhãn bằng Từ điển phân cực...")
    POS_LEX = {'tốt': 1.0, 'tuyệt': 1.5, 'tuyệt_vời': 2.0, 'xịn': 1.2, 'xịn_sò': 1.5, 'mượt': 1.2, 'mượt_mà': 1.5, 'ổn': 0.8, 'hài_lòng': 1.5, 'tự_hào': 1.8, 'yêu_thích': 1.5, 'thích': 1.0, 'mê': 1.2, 'đẹp': 1.0, 'dễ_thương': 1.0, 'đáng_yêu': 1.2, 'nhiệt_tình': 1.5, 'tận_tâm': 1.8, 'tâm_huyết': 2.0, 'dễ_hiểu': 1.5, 'chu_đáo': 1.5, 'dễ_tính': 1.2, 'chấm_dễ': 1.5, 'qua_môn': 1.2, 'đậu': 1.0, 'điểm_cao': 1.5, 'học_bổng': 1.2, 'xuất_sắc': 1.8, 'giỏi': 1.2, 'chuyên_nghiệp': 1.5, 'uy_tín': 1.5, 'tử_tế': 1.5, 'cảm_ơn': 1.2, 'biết_ơn': 1.8, 'cảm_kích': 1.8, 'vui': 1.0, 'hạnh_phúc': 1.8, 'chúc_mừng': 1.2, 'đỉnh': 1.5, 'cháy': 1.2}
    NEG_LEX = {'tệ': -1.5, 'tồi': -1.5, 'dở': -1.2, 'kém': -1.2, 'bất_cập': -1.5, 'bức_xúc': -1.8, 'chán': -1.2, 'thất_vọng': -1.8, 'bất_bình': -1.5, 'vô_lý': -1.8, 'khó_chịu': -1.5, 'chậm': -1.0, 'chậm_trễ': -1.5, 'lâu': -0.8, 'quá_tải': -1.5, 'kẹt': -1.2, 'tắc': -1.2, 'lag': -1.2, 'hỏng': -1.5, 'hư': -1.2, 'nóng': -1.0, 'bẩn': -1.5, 'rác': -1.2, 'ồn': -1.2, 'ồn_ào': -1.5, 'mất_trật_tự': -1.5, 'vô_ý_thức': -2.0, 'thiếu_ý_thức': -1.8, 'rớt': -1.5, 'rớt_môn': -1.8, 'tạch': -1.8, 'nợ_môn': -1.5, 'cấm_thi': -2.0, 'phạt': -1.2, 'khó': -0.8, 'khó_khăn': -1.2, 'áp_lực': -1.5, 'stress': -1.5, 'bế_tắc': -1.8, 'đuối': -1.2, 'bất_lực': -1.5, 'đau_khổ': -1.5, 'khóc': -1.2, 'buồn': -1.0, 'tuyệt_vọng': -2.0, 'lừa': -1.8, 'lừa_đảo': -2.0, 'chửi': -1.5}
    NEGS = {'không', 'chẳng', 'chưa', 'không_thể', 'chả', 'hổng', 'khum'}
    INTS = {'rất': 1.5, 'quá': 1.6, 'cực_kỳ': 1.8, 'siêu': 1.8, 'vô_cùng': 1.8, 'lắm': 1.4}

    def calc_sentiment(t):
        words = str(t).lower().split()
        sc, cnt = 0.0, 0
        for i, w in enumerate(words):
            is_neg = i > 0 and words[i-1] in NEGS
            m = INTS.get(words[i-1], 1.0) if i > 0 else 1.0
            val = POS_LEX.get(w, NEG_LEX.get(w, 0.0))
            if val != 0:
                if is_neg: val = -val * 0.8
                val *= m
                sc += val
                cnt += 1
        norm = sc / (cnt * 1.5) if cnt > 0 else 0.0
        if norm > 0.15: return 'POSITIVE'
        if norm < -0.15: return 'NEGATIVE'
        return 'NEUTRAL'

    src_col = 'tokenized_text' if 'tokenized_text' in df.columns else ('clean_text' if 'clean_text' in df.columns else 'raw_text')
    df['sentiment_lexicon'] = df[src_col].apply(calc_sentiment)
    label_col = 'sentiment_lexicon'
    print("-> Đã gán nhãn thành công!")

print(f"-> Phân bố nhãn trên cột '{label_col}':")
print(df[label_col].value_counts())

In [ ]:
# [BƯỚC 3] Tiền xử lý & Tokenizer PhoBERT
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

LABEL_MAP = {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}
INV_MAP = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

# Tự động phát hiện cột văn bản
text_col = 'tokenized_text' if 'tokenized_text' in df.columns else ('clean_text' if 'clean_text' in df.columns else 'raw_text')
if text_col != 'tokenized_text':
    from pyvi import ViTokenizer
    print(f"Cột '{text_col}' chưa được tách từ, đang tokenize bằng pyvi...")
    df['tokenized_text'] = df[text_col].fillna('').astype(str).apply(ViTokenizer.tokenize)
    text_col = 'tokenized_text'

df = df.dropna(subset=[text_col, label_col])
df['label_id'] = df[label_col].map(LABEL_MAP)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

texts = df[text_col].values
labels = df['label_id'].values

# Phân chia Stratified 70% Train - 15% Val - 15% Test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.30, random_state=42, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

MODEL_NAME = 'vinai/phobert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ConfessionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(ConfessionDataset(train_texts, train_labels, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(ConfessionDataset(val_texts, val_labels, tokenizer), batch_size=16, shuffle=False)
test_loader = DataLoader(ConfessionDataset(test_texts, test_labels, tokenizer), batch_size=16, shuffle=False)

In [ ]:
# [BƯỚC 4] Khởi tạo Mô hình & Trọng số bù trừ Mất cân bằng lớp (Class-Weighted Loss)
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Tính trọng số Class-Weight: lớp thiểu số (NEGATIVE) nhận trọng số cao hơn
cw = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
weights_tensor = torch.tensor(cw, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
print(f"Trọng số lớp (Class Weights): NEGATIVE={cw[0]:.2f}, NEUTRAL={cw[1]:.2f}, POSITIVE={cw[2]:.2f}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

epochs = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

In [ ]:
# [BƯỚC 5] Vòng lặp Huấn luyện (Training Loop) với Early Stopping trên Validation Macro-F1
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

best_val_f1 = 0.0
best_model_weights = None

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # Đánh giá trên Validation Set
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(targets.cpu().numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    val_p, val_r, val_f1, _ = precision_recall_fscore_support(val_targets, val_preds, average='macro', zero_division=0)
    print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_f1*100:.2f}%")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_weights = model.state_dict()
        print(f"  -> Lưu checkpoint mới nhất với Val Macro-F1: {val_f1*100:.2f}%")

# Nạp lại trọng số mô hình tốt nhất
model.load_state_dict(best_model_weights)

In [ ]:
# [BƯỚC 6] Đánh giá trên Tập kiểm thử Độc lập (Unseen Test Set - 15%)
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(batch['label'].numpy())

test_acc = accuracy_score(test_targets, test_preds)
p, r, f1, _ = precision_recall_fscore_support(test_targets, test_preds, average='macro')

print("=" * 60)
print("KẾT QUẢ CUỐI CÙNG TRÊN TẬP TEST ĐỘC LẬP (PHOBERT-BASE):")
print("=" * 60)
print(f"Accuracy:        {test_acc*100:.2f}%")
print(f"Macro-Precision: {p*100:.2f}%")
print(f"Macro-Recall:    {r*100:.2f}%")
print(f"Macro-F1:        {f1*100:.2f}%")

target_names = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
print("\nBáo cáo phân loại chi tiết từng lớp:")
print(classification_report(test_targets, test_preds, target_names=target_names))

# Trực quan hóa Ma trận nhầm lẫn
cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(6, 5), dpi=300)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Ma Trận Nhầm Lẫn PhoBERT-base (Test Set)')
plt.xlabel('Dự Đoán (Predicted)')
plt.ylabel('Thực Tế (True)')
plt.tight_layout()
plt.savefig('phobert_test_confusion_matrix.png')
plt.show()

In [ ]:
# [BƯỚC 7] Lưu mô hình & Xuất kết quả dự đoán
model.save_pretrained('phobert_uit_confession_model')
tokenizer.save_pretrained('phobert_uit_confession_model')

test_results_df = pd.DataFrame({
    'text': test_texts,
    'true_label': [INV_MAP[t] for t in test_targets],
    'predicted_label': [INV_MAP[p] for p in test_preds]
})
test_results_df.to_csv('phobert_test_predictions.csv', index=False, encoding='utf-8-sig')
print("-> Đã lưu mô hình vào: phobert_uit_confession_model/")
print("-> Đã lưu kết quả kiểm thử vào: phobert_test_predictions.csv")